# 원본 전체 불량 보고서 수정
원본 제품별 파일만 사용합니다. 실제 Run과 라벨 관측 구간을 구분하고 정상 비교를 같은 관측 구간으로 제한합니다. 직전10샷 센서 이력은 원본 비라벨까지 연결하여 다시 계산합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent
tables = root / 'reports/tables'
tables.mkdir(exist_ok=True)
features = pd.read_csv(root / 'data/moldset_labeled_cn7.csv', nrows=0).columns.drop(['Unnamed: 0','PassOrFail']).tolist()
labeled_parts, history_parts = [], []
for product in ['CN7','RG3']:
    label = pd.read_csv(root / ('data/original_split/moldset_labeled_' + product.lower() + '.csv'))
    history = pd.read_csv(root / ('data/original_split/moldset_unlabeled_' + product.lower() + '.csv'))
    label['product'] = product
    history['product'] = product
    labeled_parts.append(label)
    history_parts.append(history)
labels = pd.concat(labeled_parts, ignore_index=True)
history = pd.concat(history_parts, ignore_index=True)
for frame in [labels, history]:
    frame['TimeStamp'] = pd.to_datetime(frame['TimeStamp'])
keys = ['EQUIP_CD','product','TimeStamp']
labels['Reason'] = labels['Reason'].fillna('정상')
labels['side'] = labels['PART_NAME'].str.extract(r'(LH|RH)\s*$', expand=False)
labels['side_reason'] = labels['side'] + ' ' + labels['Reason']
def join_unique(series):
    return '·'.join(sorted(set(series.dropna().astype(str))))
label_ranges = labels.groupby(keys)[features].max() - labels.groupby(keys)[features].min()
assert not label_ranges.gt(1e-4).any(axis=1).any(), 'Labeled LH/RH sensor conflict'
label_shots = labels.groupby(keys, as_index=False).agg(
    PassOrFail=('PassOrFail','max'),
    Reason=('Reason', lambda x: join_unique(x[x.ne('정상')]) or '정상'),
    side_reason=('side_reason', join_unique)
)
# Preserve the defective side/type mapping separately, without normal side labels.
side_bad = labels.loc[labels.PassOrFail.eq(1)].groupby(keys)['side_reason'].agg(join_unique).rename('bad_side_reason')
label_shots = label_shots.drop(columns='side_reason').merge(side_bad, on=keys, how='left')
label_shots['bad_side_reason'] = label_shots['bad_side_reason'].fillna('')
label_shots = label_shots.sort_values(['product','TimeStamp']).reset_index(drop=True)
gap = label_shots.groupby('product')['TimeStamp'].diff().dt.total_seconds()
boundary = label_shots['product'].ne(label_shots['product'].shift()) | gap.gt(1800) | gap.isna()
label_shots['block_number'] = boundary.cumsum()
label_shots['comparison_block'] = label_shots.groupby('block_number')['TimeStamp'].transform('min').dt.strftime('%Y%m%d_%H%M%S') + '_' + label_shots['product']

records = pd.concat([labels[keys+features].assign(priority=0),
                     history[keys+features].assign(priority=1)], ignore_index=True)
ranges = records.groupby(keys)[features].max()-records.groupby(keys)[features].min()
conflict = ranges.gt(1e-4).any(axis=1).rename('sensor_conflict')
observed = records.sort_values('priority').drop_duplicates(keys).drop(columns='priority')
observed = observed.merge(conflict, on=keys, how='left')
label_keys = pd.MultiIndex.from_frame(label_shots[keys])
is_labeled = pd.MultiIndex.from_frame(observed[keys]).isin(label_keys)
observed.loc[observed['sensor_conflict'] & ~is_labeled, features] = np.nan
observed = observed.sort_values(['EQUIP_CD','TimeStamp','product']).reset_index(drop=True)
# A shared timestamp assigned to different products cannot define a clean machine history.
ambiguous = observed.groupby(['EQUIP_CD','TimeStamp'])['product'].transform('nunique').gt(1)
assert not pd.MultiIndex.from_frame(observed.loc[ambiguous,keys]).isin(label_keys).any()
observed = observed.loc[~ambiguous].reset_index(drop=True)
gaps = observed.groupby('EQUIP_CD')['TimeStamp'].diff().dt.total_seconds()
switch = observed['product'].ne(observed.groupby('EQUIP_CD')['product'].shift())
observed['history_segment'] = (switch | gaps.gt(1800) | gaps.isna()).cumsum()
# This segment controls rolling history; it is not an actual production Run ID.
for feature in features:
    previous = observed.groupby('history_segment')[feature].shift()
    observed[feature+'__delta1'] = observed[feature] - previous
    grouped = previous.groupby(observed['history_segment'])
    median = grouped.transform(lambda x: x.rolling(10,min_periods=5).median())
    observed[feature+'__past10_std'] = grouped.transform(lambda x: x.rolling(10,min_periods=5).std())
    observed[feature+'__dev10'] = observed[feature] - median
shots = label_shots.merge(observed, on=keys, how='left', validate='one_to_one')
assert len(shots)==1304 and shots.PassOrFail.sum()==44
assert int(labels.PassOrFail.eq(1).sum())==52
assert len(shots.comparison_block.unique())==8
shots.to_csv(tables/'original_labeled_shot_features.csv',index=False,encoding='utf-8-sig')
block_summary = shots.groupby(['product','comparison_block'],as_index=False).agg(
    start=('TimeStamp','min'),end=('TimeStamp','max'),
    labeled_shots=('PassOrFail','size'),bad_shots=('PassOrFail','sum'))
block_summary['normal_shots'] = block_summary.labeled_shots-block_summary.bad_shots
block_summary.to_csv(tables/'original_label_observation_blocks.csv',index=False,encoding='utf-8-sig')
# Each defective Shot is compared with labeled normal Shots from its own observation block.
detail=[]
stats=[]
def number(value,digits=3):
    return '비교 이력 부족' if pd.isna(value) else ('{:.'+str(digits)+'f}').format(value)
for (product, block), group in shots.groupby(['product','comparison_block']):
    normal = group[group.PassOrFail.eq(0)]
    for reason, defects in group[group.PassOrFail.eq(1)].groupby('Reason'):
        for feature in features:
            for metric in [feature,feature+'__past10_std',feature+'__dev10']:
                stats.append({'product':product,'comparison_block':block,'Reason':reason,'metric':metric,
                              'defect_n':len(defects),'normal_n':len(normal),
                              'defect_median':defects[metric].median(),'normal_median':normal[metric].median(),
                              'defect_mean':defects[metric].mean(),'normal_mean':normal[metric].mean()})
    for _, defect in group[group.PassOrFail.eq(1)].iterrows():
        row=defect.to_dict()
        parts=[]
        if '초기' in defect.Reason:
            for feature,name in [('Max_Injection_Speed','사출속도'),('Filling_Time','충전시간')]:
                parts.append('{} {} / 같은 구간 정상 중앙 {}'.format(name,number(defect[feature],2),number(normal[feature].median(),2)))
            row['comparison_note']='같은 구간의 발생 전후 정상과 설명용 비교; 정상 중앙값은 미래 정보도 포함'
        else:
            selected = [('Barrel_Temperature_2__past10_std','배럴2 흔들림'),('Hopper_Temperature__past10_std','호퍼 흔들림')] if product=='CN7' and '가스' in defect.Reason else []
            if '미성형' in defect.Reason and product=='CN7':
                selected += [('Max_Injection_Speed__dev10','속도 변화'),('Filling_Time__dev10','충전시간 변화')]
            if product=='RG3':
                selected += [('Cushion_Position__past10_std','쿠션 흔들림'),('Barrel_Temperature_5__past10_std','배럴5 흔들림'),('Barrel_Temperature_2__past10_std','배럴2 흔들림')]
            for metric,name in selected:
                center=normal[metric].median()
                row[metric+'__normal_block_median']=center
                parts.append('{} {} / 정상 {}'.format(name,number(defect[metric]),number(center)))
            row['comparison_note']='직전 최대10개 관측 Shot으로 계산한 변화/표준편차; 정상은 같은 라벨 관측 구간'
        row['sensor_observation']='; '.join(parts)
        detail.append(row)
diagnostics=pd.DataFrame(detail).sort_values(['product','TimeStamp'])
diagnostics.to_csv(tables/'all_defect_shot_diagnostics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(stats).to_csv(tables/'original_block_defect_feature_comparisons.csv',index=False,encoding='utf-8-sig')
assert len(diagnostics)==44
lines=[
    '# 원본 전체 불량: 관측 구간과 센서 변화',
    '',
    '원본 라벨·비라벨을 제품만 분리한 데이터를 사용했다. 불량 라벨은 52행이고 같은 설비·제품·Timestamp의 RH/LH를 합치면 44 Shot이다. CN7 19 Shot(초기허용불량 5 포함), RG3 25 Shot이다.',
    '',
    '## Run 구분 정정',
    '',
    '이전 보고서의 4개는 실제 Run 수가 아니다. 전체 생산 이력에서 제품 전환 또는 30분 초과 관측 공백으로 나눈 후보 구간 중 대회 라벨이 들어 있는 구간만 센 결과였다. CN7 10/16~10/20을 하나로 합쳐 서로 다른 시기의 정상값이 비교에 섞였다.',
    '',
    '이번 정상/불량 비교는 원본 라벨 관측이 30분 초과 끊기는 지점으로 나눈 8개 관측 구간(CN7 3, RG3 5)을 사용한다. 이 구간은 실제 설비 Run ID가 아니다. 비라벨 기록으로 이어지는 연속 가동과 설정 변경은 별도 확인이 필요하다. 자정만으로 구간을 나누지 않았다.',
    '',
    '| 제품 | 비교 구간 시작 | 끝 | 라벨 Shot | 정상 Shot | 불량 Shot |',
    '|---|---|---|---:|---:|---:|'
]
for b in block_summary.itertuples(index=False):
    lines.append('| {} | {} | {} | {} | {} | {} |'.format(b.product,b.start.strftime('%m/%d %H:%M'),b.end.strftime('%m/%d %H:%M'),b.labeled_shots,b.normal_shots,b.bad_shots))
lines+=['','## 불량 유형별로 보인 변화','','값이 높은지보다 이전 Shot들의 변화량과 흔들림을 함께 본다. 아래 흔들림은 직전 최대 10개 관측 Shot의 표준편차이며 최소 5개의 유효 관측이 필요하다. 각 비교값은 해당 불량 유형과 같은 구간 정상 Shot에서 구한 중앙값이다.']
for (product,block),g in shots.groupby(['product','comparison_block']):
    normal=g[g.PassOrFail.eq(0)]
    for reason,bad in g[g.PassOrFail.eq(1)].groupby('Reason'):
        fs=[('Barrel_Temperature_2__past10_std','배럴2 흔들림'),('Hopper_Temperature__past10_std','호퍼 흔들림')] if product=='CN7' and '가스' in reason else []
        if product=='CN7' and '미성형' in reason:
            fs += [('Max_Injection_Speed__dev10','속도 변화'),('Filling_Time__dev10','충전시간 변화')]
        if '초기' in reason:
            fs=[('Max_Injection_Speed','사출속도'),('Filling_Time','충전시간')]
        if product=='RG3':
            fs=[('Cushion_Position__past10_std','쿠션 흔들림'),('Barrel_Temperature_5__past10_std','배럴5 흔들림')]
        text='; '.join('{}: 불량 {} / 정상 {}'.format(name,number(bad[f].median()),number(normal[f].median())) for f,name in fs)
        lines+=['','- {} {} 시작 구간, {} {} Shot — {}'.format(product,pd.to_datetime(block[:15],format='%Y%m%d_%H%M%S').strftime('%m/%d %H:%M'),reason,len(bad),text)]
lines+=['','## 모든 불량 Shot','','각 행에서 불량 Shot의 값과 같은 관측 구간 정상 중앙값을 나란히 표시했다. 이 표는 설명용이며 분류 성능이나 불량 원인 확정이 아니다.','',
        '| 제품 | 시각 | 좌우별 불량 | 비교 구간 시작 | 센서값 / 같은 구간 정상 |','|---|---|---|---|---|']
for b in diagnostics.itertuples(index=False):
    block_time=pd.to_datetime(b.comparison_block[:15],format='%Y%m%d_%H%M%S').strftime('%m/%d %H:%M')
    lines.append('| {} | {} | {} | {} | {} |'.format(b.product,b.TimeStamp.strftime('%m/%d %H:%M:%S'),b.bad_side_reason,block_time,b.sensor_observation))
lines+=['','## 해석 범위','',
    '- 비라벨 데이터에는 정답이 없다. 직전 센서 이력에 사용했지만 정상으로 간주하지 않았다.',
    '- 최근 표준편차가 정상 중앙값보다 높다는 사실만으로 이상치라고 판정하지 않는다. 정상/불량 분포의 겹침과 새 구간에서의 재현을 따로 평가해야 한다.',
    '- 초기허용불량은 10/27 원본 라벨에만 있다. 해당 구간은 00:10부터 시작하고, 첫 불량 이전 정상 26 Shot과 마지막 불량 이후 정상 76 Shot이 있다. 라벨 명칭만으로 실제 가동 시작 직후라고 단정하지 않는다. 정상 102 Shot 중앙값에는 발생 전후 정보가 포함되므로 설명용 비교에만 쓴다.',
    '- 센서 단위가 확인되지 않은 값은 파일에 기록된 숫자로 제시했다. 표준화된 대회용 값과 혼용하지 않는다.',
    '- 실제 Run 경계는 작업 기록·설정값 변경 이력이 필요하다. 원본 전체의 Run 수를 4개 또는 8개로 확정하지 않는다.']
report='\n'.join(lines)+'\n'
(root/'reports/all_defect_run_context_summary.md').write_text(report,encoding='utf-8')
(root/'reports/all_defect_shot_summary.md').write_text(report,encoding='utf-8')
print(block_summary.to_string(index=False))
print('Verified: labeled rows',len(labels),'shots',len(shots),'bad rows',52,'bad shots',len(diagnostics))
print('Reports and detailed tables replaced with original-source analysis.')
